<a href="https://colab.research.google.com/github/litlig/notebooks/blob/main/discrete_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build language generation using discrete diffusion

Diffusion models are largely used in image and video generation. It operates in high-dim continuous space, where we convert noise to image following a denoise process. On the other hand, text generation are discrete in nature. For a text sequence, we tokenize it and get a vector of token index. The numberical difference between tokens have no natural meaning, it does not mean a token is semantically closer to one or other.

Discrete diffusion model is designed for text gen. Here we use the Shakespear dataset from Andrej Karparthy's LLM series, build a discrete DiT model from scratch and see if it can generate Shakespear like text sequence.


In [1]:
#@title import dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

print("-"*10 + "text sample" + "-"*60)
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))
print(text[:200])
print('\n')

vocab = sorted(list(set(text)))
vocab_size = len(vocab)
print("-"*10 + "vocab by character" + "-"*60)
print("vocab size: ", len(vocab))
print("vocab: ", vocab)

stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

--2026-07-17 21:18:57--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-07-17 21:18:57 (33.2 MB/s) - ‘input.txt’ saved [1115394/1115394]

----------text sample------------------------------------------------------------
length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


----------vocab by character----------------------------------------------

### CTMC and rate matrix
To recall, the continuous diffusion model is done by flow matching, where it defines a vector field $𝑢_t$ which tells the direction and velocity a noised image $x_t$ should move at time t. Since we do not know the distribution of image manifold, $u_t$ is not tractable. However, if know the final state (a sample image), $u_t$ can be calculated analytically; one solution would be linear connecting the noised point $x_t$ and the final state $z$. However, what we need is not the vector field for a given $z$, but the average vector field for all z that follows the posterior distribution $p(z|x_t)$. In neural nets training, the loss is always the average against all samples. By training a nn that minize the average loss against the conditional vector field, we also get the marginal vector field we need.

For a text sequence, we tokenize it and get a vector of token index. The transition from noise to meaningful text sequence is jumping across discrete states. Instead of vector field, we define a **rate matrix** $Q_t(y|x)$, the rate of jumping from state x to state y at time t. Suppose we are at state x at time t, given a very small time interval $h$, the probability we jump to $y(y \neq x)$ is $h Q_t(y|x)$.

Now suppose we have the rate matrix Q, how to transite a sampled noise to meanful texts?

The number of states is exponential to the sequence length, $V^d$ wehre $V$ is the vocab size and d is the sequence length. To reduce the size of Q, the rate is 0 is more than one position is changed. For $X_t=x$, a rate matrix $Q_t(v,j) defines the rate of change at position j to token v, we can design a sampling algorithm with Euler approximation.


In [2]:
from abc import ABC, abstractmethod
import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

class NoiseGen(ABC):
  @abstractmethod
  def gen(batch_size, n_seq):
    pass

def euler_step(x_t, rates, h):
    # delta_{v,x}: (B, d, V) one-hot at the current token
    delta = F.one_hot(x_t, num_classes=rates.size(-1)).to(rates.dtype)

    off_diag = h * rates * (1.0 - delta)          # h*q(v) for v != x, else 0
    stay = (1.0 - off_diag.sum(-1, keepdim=True)) # 1 - h*sum_{v!=x} q(v)

    probs = off_diag + delta * stay

    return torch.distributions.Categorical(probs=probs).sample()

def sample(model, n_seq, n_steps, noise_gen):
  ts = torch.linspace(0.0, 1.0, n_steps + 1, device=device)
  x = noise_gen.gen(1, n_seq)
  for i in range(n_steps):
    s, t = ts[i], ts[i + 1]
    rate_mtx = model(x, s) # shape n_seq:n_vocab
    x = euler_step(x, rate_mtx, t - s)
  return x

In [3]:
#@title sample without training

class UniformNoiseGen(NoiseGen):
  def gen(self, batch_size, n_seq):
    return torch.randint(0, vocab_size, (batch_size, n_seq), device=device)

class MockModel(nn.Module):
  def forward(self, x, t):
    logits = torch.randn(x.shape[-1], vocab_size, device=device)
    return F.softmax(logits, dim=-1) - F.one_hot(x, num_classes=vocab_size).to(device)

model = MockModel()
noise_gen = UniformNoiseGen()

x = sample(model, 128, 100, noise_gen)
[decode(row.tolist()) for row in x]

["nwAlQgrsSo 'bmm,DjUmT?hkdwWpui:&N\niVXMEEqDm'zkh\n3FfL?EK,oAvGkdLAK!x,cX.bG \nYEaQcTAyX\nXBRtrp.amgH3DpJdHPZnY&o.XLAHEyNyKEoWkNYis.u"]

### Factorized mixture path
Given the initial noise distribution of $p_{init}$ and the final data distribution of $p_{data}$, a discrete probability path is $p_t$ such that $p_0$ ~ $p_{init}$ and $p_1$ ~ $p_{data}$.

The most commonly used discrete probability path is factorized mixture path, which defines the conditional path as:
$$p_t(x|z) = ∏_{j=1}^d [(1- κ_t) \,p_{\mathrm{init}}^{(j)}(x_j) + \kappa_t\,\delta_{z_j}(x_j)]$$
wehre $κ_t$ is the noise schedule.

We can see each token position are calculated independently. The rate matrix conditioned on z:
$$Q^z_t(i,v|x_i) = \frac {\dotκ_t} {1-κ_t}(\delta_{z_i}(v) - δ_{x_i}(v))$$

The marginal rate matrix
$$Q_t(i,v|x_i) = Σ_z Q^z_t(i,v|x_i) p(z|x) = \frac {\dotκ_t} {1-κ_t}(p(z_j=v|x) - δ_{x_i}(v))$$

The problem now becomes categorization of z given x and t, and the rate matrix is a reparameterization of it.

In [31]:
#@title Transformer model definition
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    """Standard transformer-style sinusoidal embedding for a scalar t."""

    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device).float() / half
        )

        args = (t.float()*1000).unsqueeze(-1) * freqs.unsqueeze(0)  # (B, half)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return emb

class TimestepMLP(nn.Module):
    """Sinusoidal embedding -> MLP -> conditioning vector c."""

    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.sinusoidal = SinusoidalTimeEmbedding(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.sinusoidal(t))  # (B, cond_dim)


def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class AdaLNZeroBlock(nn.Module):
    """
    Transformer encoder block with AdaLN-Zero conditioning on t.

    Six modulation params per block (as in DiT):
      shift_msa, scale_msa, gate_msa   -> for the attention sub-layer
      shift_mlp, scale_mlp, gate_mlp   -> for the MLP sub-layer

    The final Linear producing these params is zero-initialized so each
    block starts as an identity function (gate = 0), which stabilizes
    training at initialization.
    """

    def __init__(self, dim: int, num_heads: int, cond_dim: int, mlp_ratio: float = 4.0,
                 dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 6 * dim),
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x: torch.Tensor, c: torch.Tensor,
                attn_mask: torch.Tensor = None,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        (shift_msa, scale_msa, gate_msa,
         shift_mlp, scale_mlp, gate_mlp) = self.adaLN_modulation(c).chunk(6, dim=-1)

        h = modulate(self.norm1(x), shift_msa, scale_msa)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        x = x + gate_msa.unsqueeze(1) * attn_out

        h = modulate(self.norm2(x), shift_mlp, scale_mlp)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(h)
        return x


class FinalAdaLN(nn.Module):
    """Final norm + modulation before projecting to vocab logits."""

    def __init__(self, dim: int, cond_dim: int, vocab_size: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * dim),
        )
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)
        self.head = nn.Linear(dim, vocab_size)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=-1)
        x = modulate(self.norm(x), shift, scale)
        return self.head(x)  # (B, L, vocab_size)

class DiscreteDiffusionTransformer(nn.Module):
    """
    Input:  x_t (B, L) token ids (possibly containing a [MASK] id), t (B,)
    Output: logits (B, L, vocab_size) predicting the denoised/original tokens
    """

    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int = 64,
        dim: int = 64,
        depth: int = 4,
        num_heads: int = 4,
        cond_dim: int = 64,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len, dim))
        nn.init.normal_(self.pos_emb, std=0.02)

        self.time_mlp = TimestepMLP(hidden_dim=cond_dim, cond_dim=cond_dim)

        self.blocks = nn.ModuleList([
            AdaLNZeroBlock(dim, num_heads, cond_dim, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.final = FinalAdaLN(dim, cond_dim, vocab_size)

    def forward(
        self,
        x_t: torch.Tensor,          # (B, L) long
        t: torch.Tensor,            # (B,) float or long
        padding_mask: torch.Tensor = None,  # (B, L) bool, True = PAD (ignored)
    ) -> torch.Tensor:
        B, L = x_t.shape
        x = self.token_emb(x_t) + self.pos_emb[:, :L, :]
        c = self.time_mlp(t)  # (B, cond_dim) — conditioning vector shared across layers

        for block in self.blocks:
            x = block(x, c, key_padding_mask=padding_mask)

        logits = self.final(x, c)
        return logits

In [32]:
#@title Training to predict z
import torch
import torch.nn.functional as F

from torch.distributions import Bernoulli

data = torch.tensor(encode(text), dtype=torch.long, device=device)

n_sample = 32
n_seq = 64

def schedule(t):
  return t

def get_batch():
  t = torch.rand(n_sample, device=device)
  kappa = schedule(t)
  seq_idx = torch.randint(0, len(text) - n_seq + 1, (n_sample,), device=device).unsqueeze(1) + torch.arange(n_seq, device=device)
  z = data[seq_idx] # sample:seq
  masks = torch.bernoulli(kappa.unsqueeze(1).expand(-1, n_seq)).long() # sample:seq
  noise = noise_gen.gen(n_sample, n_seq)
  x = masks * z + (1-masks) * noise
  return x, t, z

def loss_fn(z_pred, z):
  return F.cross_entropy(z_pred.view(-1, vocab_size), z.view(-1))

model = DiscreteDiffusionTransformer(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), 1e-3)

In [38]:
losses = []
num_epochs = 10000

print(device)
for epoch in range(num_epochs):
  optimizer.zero_grad()
  x, t, z = get_batch()
  #padding_mask = x == 0

  logits = model(x, t)
  loss = loss_fn(logits, z)
  loss.backward()
  optimizer.step()

  if epoch % 1000 == 0:
    print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    losses.append(loss.item())

cuda
Epoch 0, Loss: 1.625237
Epoch 1000, Loss: 1.653322
Epoch 2000, Loss: 1.673059
Epoch 3000, Loss: 1.760014
Epoch 4000, Loss: 1.466388
Epoch 5000, Loss: 1.611646
Epoch 6000, Loss: 1.428862
Epoch 7000, Loss: 1.724199
Epoch 8000, Loss: 1.592783
Epoch 9000, Loss: 1.861723


In [39]:
#@title Sample with trained model
@torch.no_grad()
def sample(model, n_seq, n_steps, noise_gen):
  model.eval()
  ts = torch.linspace(0.0, 1.0, n_steps + 1, device=device)
  x = noise_gen.gen(1, n_seq)
  for i in range(n_steps):
    s, t = ts[i], ts[i + 1]
    logits = model(x, s.expand(x.shape[0])) # shape n_seq:n_vocab
    rate_mtx = (F.softmax(logits, dim=-1) - F.one_hot(x, num_classes=vocab_size).to(device)) / (1-s)
    x = euler_step(x, rate_mtx, t - s)
  return x

x = sample(model, 64, 100, noise_gen)
[print(decode(row.tolist())) for row in x]

d did neyce?

HONRY VIA: sutur selon'sl mave,
Myoury abenly tuke


[None]